# REQ_117 Phase 1a — Activation DMD Signal Check

Cheap inspection point for the windowed + per-regime activation DMD analyzer.
Phase 1b (views, dashboard, validation write-up) is conditional on what these
plots show.

## What we're looking for

Initial hypothesis was *eigenvalue migration to conjugate pairs on the unit
circle*. Canon and p109 show a different story: eigenvalues stay parked at
(1, 0); the diagnostic signal lives in **residual norm structure** — where
the linear approximation breaks down identifies regime boundaries.

This peek extends the inspection to all four reference variants before we
tune the regime-detection primitive. Two known structural facts inform
what to look for:

- **p109 (clean grokker)** has *nearly-aligned saddle transits* with low
  orthogonality in early training, plus evidence that something else is
  poised to happen at the very end (late-training boundaries are
  anticipated, not threshold artifacts).
- **p101 (degenerate)** has *multi-segment orthogonal drift* — repeated
  reorganization across distinct geometric directions. We should expect
  more visible regime structure than p109.

## Reference variants

| Variant | Role | Expected signature |
|---|---|---|
| `p113/s999/ds598` | Canon | Slow / diffuse grokking; wide bump centered at ~10k |
| `p109/s485/ds598` | Clean grokker | Sharp early reorganization; aligned saddle transits |
| `p101/s999/ds598` | Degenerate | Multi-segment orthogonal drift; more regime structure than p109 |
| `p59/s485/ds598`  | Failure | Qualitatively different — wandering or unstructured |

## Sites

Per-site windowed DMD: `resid_pre`, `attn_out`, `mlp_out`, `resid_post`.
Each site has its own residual time series, regime boundaries, and per-regime
spectra — sites move on different timelines.
## Exports

Each plot is saved to `apps/research/exports/req_117/{name}.png` as it is
displayed inline. File names match the plot's role (e.g. `residual_p113.png`,
`cross_variant_mlp_out.png`).

## Setup

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

from miscope import load_family

SITES = ['resid_pre', 'attn_out', 'mlp_out', 'resid_post']
SITE_COLORS = {
    'resid_pre':  '#1f77b4',
    'attn_out':   '#2ca02c',
    'mlp_out':    '#d62728',
    'resid_post': '#9467bd',
}
VARIANT_COLORS = {
    'p113/s999/ds598': '#1f77b4',  # canon — blue
    'p109/s485/ds598': '#2ca02c',  # clean grokker — red
    'p101/s999/ds598': '#9467bd',  # degenerate — purple
    'p59/s485/ds598':  '#8c564b',  # failure — brown
}

VARIANTS = [
    {'label': 'p113/s999/ds598', 'role': 'canon',
     'parameters': {'prime': 113, 'seed': 999, 'data_seed': 598}},
    {'label': 'p109/s485/ds598', 'role': 'clean grokker',
     'parameters': {'prime': 109, 'seed': 485, 'data_seed': 598}},
    {'label': 'p101/s999/ds598', 'role': 'degenerate',
     'parameters': {'prime': 101, 'seed': 999, 'data_seed': 598}},
    {'label': 'p59/s485/ds598',  'role': 'failure',
     'parameters': {'prime': 59,  'seed': 485, 'data_seed': 598}},
]
ALL_LABELS = [v['label'] for v in VARIANTS]
FOCUS = 'p113/s999/ds598'

EXPORT_DIR = Path('../exports/req_117')
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

def show_and_export(fig, name):
    """Save figure to EXPORT_DIR/name.png and return for inline display."""
    fig.write_image(str(EXPORT_DIR / f'{name}.png'), scale=2)
    return fig

In [ ]:
family = load_family('modulo_addition_1layer')

def load_activation_dmd(label):
    spec = next(v for v in VARIANTS if v['label'] == label)
    p = spec['parameters']
    variant = family.get_variant(prime=p['prime'], seed=p['seed'], data_seed=p['data_seed'])
    return variant.artifacts.load_cross_epoch('activation_dmd')

data = {}
for v in VARIANTS:
    try:
        data[v['label']] = load_activation_dmd(v['label'])
    except FileNotFoundError as e:
        print(f'!! {v["label"]}: {e}')
        print(f'   Run: uv run python scripts/run_analysis.py --variant {v["label"].replace("/","_")} --analyzer activation_dmd')

# Quick summary across loaded variants
for label, d in data.items():
    print(f'{label}:')
    print(f'  epochs: {len(d["epochs"])}, range [{int(d["epochs"][0])}, {int(d["epochs"][-1])}]')
    for site in SITES:
        n_windows = len(d[f'{site}__windowed__window_starts'])
        n_regimes = len(d[f'{site}__regimes__segment_starts'])
        n_tracks = int(d[f'{site}__tracks__n_tracks'])
        threshold = float(d[f'{site}__regimes__threshold_used'])
        res_range = (float(d[f'{site}__windowed__residual_norm_mean'].min()),
                     float(d[f'{site}__windowed__residual_norm_mean'].max()))
        print(f'    {site:<10}: {n_windows} windows | {n_regimes} regimes | '
              f'{n_tracks} tracks | threshold={threshold:.3f} | '
              f'residual range=[{res_range[0]:.3f}, {res_range[1]:.3f}]')

## 1. Residual norm trajectory with regime boundaries

Per-site mean residual norm across windows, with regime boundaries
overlaid as vertical dashed lines. The primary diagnostic.

**What to check:**
- p101 vs. p109: does p101 show more (or differently-spaced) regime
  structure consistent with the multi-segment orthogonal drift hypothesis?
- p59: qualitatively different residual character — wandering, no clear
  basins, sustained noise floor?
- Late-training p109 boundaries: real signal (not noise to be tuned away).

In [ ]:
def plot_residuals_with_regimes(d, label):
    fig = make_subplots(
        rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.04,
        subplot_titles=SITES,
    )
    epochs = d['epochs']
    for i, site in enumerate(SITES):
        starts = d[f'{site}__windowed__window_starts']
        ends = d[f'{site}__windowed__window_ends']
        centers = (starts + ends) // 2
        x = epochs[centers]
        y = d[f'{site}__windowed__residual_norm_mean']
        threshold = float(d[f'{site}__regimes__threshold_used'])

        fig.add_trace(
            go.Scatter(x=x, y=y, mode='lines', name=site,
                       line=dict(color=SITE_COLORS[site], width=1.5)),
            row=i+1, col=1,
        )
        fig.add_hline(
            y=threshold, line_dash='dot', line_color='gray',
            annotation_text=f'threshold={threshold:.2f}', annotation_position='top right',
            row=i+1, col=1,
        )
        for b in d[f'{site}__regimes__boundary_indices']:
            b_epoch = epochs[centers[int(b)]]
            fig.add_vline(
                x=b_epoch, line_dash='dash', line_color='black',
                opacity=0.5, row=i+1, col=1,
            )
        fig.update_yaxes(title_text='residual', row=i+1, col=1)
    fig.update_xaxes(title_text='epoch', row=4, col=1)
    fig.update_layout(
        title=f'{label} — windowed DMD residual + regime boundaries',
        height=900, showlegend=False,
    )
    return fig

show_and_export(plot_residuals_with_regimes(data[FOCUS], FOCUS), 'residual_p113')

In [ ]:
show_and_export(plot_residuals_with_regimes(data['p109/s485/ds598'], 'p109/s485/ds598'), 'residual_p109')

In [ ]:
show_and_export(plot_residuals_with_regimes(data['p101/s999/ds598'], 'p101/s999/ds598'), 'residual_p101')

In [ ]:
show_and_export(plot_residuals_with_regimes(data['p59/s485/ds598'], 'p59/s485/ds598'), 'residual_p59')

## 2. Eigenvalue migration in the complex plane

All eigenvalues per window plotted on the complex plane, colored by
training time. Unit circle overlay marks the boundary between decaying
(inside) and oscillatory-stable (on circle).

Canon and p109 already showed this lens is *less* informative than the
residual lens — eigenvalues parked at (1, 0). For p101 and p59, we're
checking whether their richer regime structure surfaces any eigenvalue
migration that canon and p109 lacked, or whether the activation centroid
dynamics are universally slow drift across all four variants.

In [ ]:
def plot_eigenvalue_migration(d, label):
    fig = make_subplots(
        rows=2, cols=2, subplot_titles=SITES,
        horizontal_spacing=0.10, vertical_spacing=0.12,
    )
    epochs = d['epochs']
    for i, site in enumerate(SITES):
        row, col = (i // 2) + 1, (i % 2) + 1
        # Unit circle as a shape so it does not constrain auto-range.
        # Auto-range zooms to the actual eigenvalue cluster, surfacing the
        # small-radius migration structure that's invisible at the full
        # ±1.5 view.
        fig.add_shape(
            type='circle', xref='x domain', yref='y domain',
            x0=0, y0=0, x1=1, y1=1,
            line=dict(color='rgba(0,0,0,0)'),  # invisible — only here for layout
            row=row, col=col,
        )
        # Real unit circle scatter, but with `cliponaxis=True` so it does not
        # extend the auto-range. Tradeoff: the unit circle may be partially
        # off-screen at zoom — that's expected when eigenvalues sit near (1,0).
        theta = np.linspace(0, 2*np.pi, 200)
        cx, cy = np.cos(theta), np.sin(theta)
        fig.add_trace(
            go.Scatter(
                x=cx, y=cy, mode='lines',
                line=dict(color='lightgray', width=1),
                hoverinfo='skip', showlegend=False,
                cliponaxis=True,
            ),
            row=row, col=col,
        )
        eigs = d[f'{site}__windowed__eigenvalues']
        n_modes_per_window = d[f'{site}__windowed__n_modes_per_window']
        starts = d[f'{site}__windowed__window_starts']
        ends = d[f'{site}__windowed__window_ends']
        center_epochs = epochs[(starts + ends) // 2]
        all_real, all_imag, all_epoch = [], [], []
        for w_idx in range(len(starts)):
            k = int(n_modes_per_window[w_idx])
            valid = eigs[w_idx, :k]
            all_real.extend(valid.real.tolist())
            all_imag.extend(valid.imag.tolist())
            all_epoch.extend([int(center_epochs[w_idx])] * k)
        # Auto-range to the eigenvalue points themselves, with a small pad.
        if all_real:
            r_lo, r_hi = min(all_real), max(all_real)
            i_lo, i_hi = min(all_imag), max(all_imag)
            r_pad = max((r_hi - r_lo) * 0.10, 0.05)
            i_pad = max((i_hi - i_lo) * 0.10, 0.05)
            x_range = [r_lo - r_pad, r_hi + r_pad]
            y_range = [i_lo - i_pad, i_hi + i_pad]
        else:
            x_range = [-1.5, 1.5]
            y_range = [-1.5, 1.5]
        fig.add_trace(
            go.Scatter(
                x=all_real, y=all_imag, mode='markers',
                marker=dict(
                    size=4, color=all_epoch, colorscale='Viridis',
                    showscale=(i == 0),
                    colorbar=dict(title='epoch', len=0.4) if i == 0 else None,
                ),
                showlegend=False, name=site,
                cliponaxis=True,
            ),
            row=row, col=col,
        )
        fig.update_xaxes(title_text='Re(λ)', row=row, col=col,
                         range=x_range, zeroline=True)
        fig.update_yaxes(title_text='Im(λ)', row=row, col=col,
                         range=y_range, zeroline=True)
    fig.update_layout(
        title=f'{label} — eigenvalue migration across training (auto-zoom)',
        height=850, width=900,
    )
    return fig

show_and_export(plot_eigenvalue_migration(data[FOCUS], FOCUS), 'eigenvalues_p113')

In [ ]:
show_and_export(plot_eigenvalue_migration(data['p109/s485/ds598'], 'p109/s485/ds598'), 'eigenvalues_p109')

In [ ]:
show_and_export(plot_eigenvalue_migration(data['p101/s999/ds598'], 'p101/s999/ds598'), 'eigenvalues_p101')

In [ ]:
show_and_export(plot_eigenvalue_migration(data['p59/s485/ds598'], 'p59/s485/ds598'), 'eigenvalues_p59')

## 3. Per-track magnitude and angle trajectories

Each tracked mode plotted as a time series: `|λ|` on top, `arg(λ)` on
bottom. For canon and p109 these were dominated by stable parked tracks;
the question is whether p101 and p59 show more dynamic track behavior.

In [ ]:
def plot_track_trajectories(d, label, site):
    eigs = d[f'{site}__windowed__eigenvalues']
    track_ids = d[f'{site}__tracks__track_ids']
    n_modes_per_window = d[f'{site}__windowed__n_modes_per_window']
    starts = d[f'{site}__windowed__window_starts']
    ends = d[f'{site}__windowed__window_ends']
    epochs = d['epochs']
    center_epochs = epochs[(starts + ends) // 2]
    n_tracks = int(d[f'{site}__tracks__n_tracks'])

    track_data = {t: {'epoch': [], 'eig': []} for t in range(n_tracks)}
    for w in range(len(starts)):
        k = int(n_modes_per_window[w])
        for slot in range(k):
            tid = int(track_ids[w, slot])
            if tid >= 0:
                track_data[tid]['epoch'].append(int(center_epochs[w]))
                track_data[tid]['eig'].append(complex(eigs[w, slot]))

    fig = make_subplots(
        rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.06,
        subplot_titles=[f'|λ| per track', f'arg(λ) per track (radians)'],
    )
    for tid, td in track_data.items():
        if not td['epoch']:
            continue
        eig_arr = np.array(td['eig'])
        fig.add_trace(
            go.Scatter(x=td['epoch'], y=np.abs(eig_arr), mode='lines',
                       name=f'track {tid}', legendgroup=f't{tid}'),
            row=1, col=1,
        )
        fig.add_trace(
            go.Scatter(x=td['epoch'], y=np.angle(eig_arr), mode='lines',
                       name=f'track {tid}', legendgroup=f't{tid}', showlegend=False),
            row=2, col=1,
        )
    fig.add_hline(y=1.0, line_dash='dot', line_color='gray', row=1, col=1)
    fig.add_hline(y=0.0, line_dash='dot', line_color='gray', row=2, col=1)
    fig.update_yaxes(title_text='|λ|', row=1, col=1)
    fig.update_yaxes(title_text='arg(λ)', row=2, col=1, range=[-np.pi, np.pi])
    fig.update_xaxes(title_text='epoch', row=2, col=1)
    fig.update_layout(
        title=f'{label} — site={site} — eigenvalue track trajectories',
        height=600,
    )
    return fig

show_and_export(plot_track_trajectories(data[FOCUS], FOCUS, 'mlp_out'), 'tracks_p113_mlp_out')

In [ ]:
show_and_export(plot_track_trajectories(data['p109/s485/ds598'], 'p109/s485/ds598', 'mlp_out'), 'tracks_p109_mlp_out')

In [ ]:
show_and_export(plot_track_trajectories(data['p101/s999/ds598'], 'p101/s999/ds598', 'mlp_out'), 'tracks_p101_mlp_out')

In [ ]:
show_and_export(plot_track_trajectories(data['p59/s485/ds598'], 'p59/s485/ds598', 'mlp_out'), 'tracks_p59_mlp_out')

## 4. Per-regime DMD residual — does the recursive pass help?

Comparison of windowed mean residual vs. per-regime DMD mean residual.
Per-regime is currently a single linear operator fit across each segment;
the dashed black line shows its average per-step error within the segment.

**Caveat:** the per-regime numbers depend on segment quality. p109's
currently-too-sensitive segmentation (Problem B) and `resid_post`'s
missing-boundary issue (Problem A) will distort this comparison until
the detection primitive is refined.

In [ ]:
def plot_per_regime_vs_windowed(d, label):
    fig = make_subplots(
        rows=2, cols=2, subplot_titles=SITES,
        horizontal_spacing=0.12, vertical_spacing=0.12,
    )
    epochs = d['epochs']
    for i, site in enumerate(SITES):
        row, col = (i // 2) + 1, (i % 2) + 1
        w_starts = d[f'{site}__windowed__window_starts']
        w_ends = d[f'{site}__windowed__window_ends']
        w_residual = d[f'{site}__windowed__residual_norm_mean']
        w_x = epochs[(w_starts + w_ends) // 2]
        pr_starts = d[f'{site}__per_regime__segment_starts']
        pr_ends = d[f'{site}__per_regime__segment_ends']
        pr_residual = d[f'{site}__per_regime__residual_norm_mean']
        fig.add_trace(
            go.Scatter(x=w_x, y=w_residual, mode='lines',
                       line=dict(color=SITE_COLORS[site], width=1),
                       name=f'{site} windowed', showlegend=(i == 0)),
            row=row, col=col,
        )
        for s, e, r in zip(pr_starts, pr_ends, pr_residual):
            if np.isnan(r):
                continue
            x_seg = [int(epochs[s]), int(epochs[min(e - 1, len(epochs) - 1)])]
            fig.add_trace(
                go.Scatter(x=x_seg, y=[r, r], mode='lines',
                           line=dict(color='black', width=2, dash='dash'),
                           name='per-regime' if (i == 0 and s == pr_starts[0]) else '',
                           showlegend=(bool(i == 0 and s == pr_starts[0]))),
                row=row, col=col,
            )
        fig.update_yaxes(title_text='mean residual', row=row, col=col, type='log')
        fig.update_xaxes(title_text='epoch', row=row, col=col)
    fig.update_layout(
        title=f'{label} — windowed vs. per-regime mean residual (log y)',
        height=750, width=950,
    )
    return fig

show_and_export(plot_per_regime_vs_windowed(data[FOCUS], FOCUS), 'per_regime_p113')

In [ ]:
show_and_export(plot_per_regime_vs_windowed(data['p109/s485/ds598'], 'p109/s485/ds598'), 'per_regime_p109')

In [ ]:
show_and_export(plot_per_regime_vs_windowed(data['p101/s999/ds598'], 'p101/s999/ds598'), 'per_regime_p101')

In [ ]:
show_and_export(plot_per_regime_vs_windowed(data['p59/s485/ds598'], 'p59/s485/ds598'), 'per_regime_p59')

## 5. Cross-variant residual comparison

All four variants overlaid on the same axes per site. The strongest
single-image read on whether the residual lens distinguishes the four
variants in ways that match their known structural roles.

In [ ]:
def cross_variant_residual(labels, site='mlp_out'):
    fig = go.Figure()
    for label in labels:
        if label not in data:
            continue
        d = data[label]
        starts = d[f'{site}__windowed__window_starts']
        ends = d[f'{site}__windowed__window_ends']
        x = d['epochs'][(starts + ends) // 2]
        y = d[f'{site}__windowed__residual_norm_mean']
        color = VARIANT_COLORS.get(label, '#444')
        fig.add_trace(go.Scatter(x=x, y=y, mode='lines', name=label,
                                 line=dict(color=color, width=1.5)))
        for b in d[f'{site}__regimes__boundary_indices']:
            fig.add_vline(
                x=int(d['epochs'][(starts[int(b)] + ends[int(b)]) // 2]),
                line_dash='dash', line_color=color, opacity=0.3,
            )
    fig.update_layout(
        title=f'{site} mean residual — {", ".join(labels)}',
        xaxis_title='epoch', yaxis_title='mean residual (log)',
        yaxis_type='log', height=500,
    )
    return fig

show_and_export(cross_variant_residual(ALL_LABELS, site='mlp_out'), 'cross_variant_mlp_out')

In [ ]:
show_and_export(cross_variant_residual(ALL_LABELS, site='resid_post'), 'cross_variant_resid_post')

In [ ]:
show_and_export(cross_variant_residual(ALL_LABELS, site='attn_out'), 'cross_variant_attn_out')

In [ ]:
show_and_export(cross_variant_residual(ALL_LABELS, site='resid_pre'), 'cross_variant_resid_pre')

## Observations (fill in after inspection)

Use this section to record what the plots actually show across all four
variants. Decision is whether to:
1. Refine `detect_regime_boundaries` (peak detection + minimum exceedance
   duration) and re-run.
2. Proceed to phase 1b with the current detection.
3. Stop — residual story doesn't hold across the spectrum.

### Cross-variant pattern (the load-bearing question)
- Does p101 show more multi-segment regime structure than p109 (consistent
  with the orthogonal-drift saddle pattern)?
- Does p59 look qualitatively different from the three grokkers, or just
  noisier?
- Do the late-training p109 boundaries appear in any other variant?

### Per-variant notes
- p113 (canon): …
- p109 (clean grokker): …
- p101 (degenerate): …
- p59 (failure): …

### Detection-tuning implications
- Problem A (signal-starts-above-threshold): how often does this come up
  across the four variants? Most common at which sites?
- Problem B (over-sensitive late-training threshold): how prevalent across
  variants?
- Are there *new* failure modes visible in p101 or p59 that we hadn't
  anticipated?

### Decision
- [ ] Refine detection (option 1 + possibly 2), re-run, re-inspect
- [ ] Proceed to phase 1b with current detection
- [ ] Stop